# KCIA v134 — WAN PT (training-first publication cadence)

4 minibatches/event → 32 minibatches/event. ReLoRA refresh remains every 20 signed events. Train during snapshot sync without dropping signed events.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()   # KCIA Cloudflare release ZIP を選択
assert len(uploaded)==1 and next(iter(uploaded)).endswith('.zip')
source = Path(next(iter(uploaded)))
root = Path('/content/kcia')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(source) as z:
    for entry in z.infolist():
        dest=(root/entry.filename).resolve()
        if not dest.is_relative_to(root.resolve()):
            raise ValueError('unsafe archive entry')
    z.extractall(root)
assert (root/'scripts/check_wan_p2p.py').exists()
print('KCIA sources extracted:',root)


In [ ]:
%cd /content/kcia
%pip -q install -r deploy/vps/requirements-transport.txt


In [ ]:
import sys, subprocess
subprocess.run([sys.executable, 'scripts/wan_preflight.py', '--role', 'colab', '--require-deps'], check=True)


**設定**: Cloudflareで公開するシグナリング用ホスト名だけを指定します。クライアント用トークンは不要です。Colabに秘密鍵を持ち込まないでください。

In [ ]:
signal_host = input('Cloudflare signal hostname (signal.example.com): ').strip()
assert signal_host and '://' not in signal_host
signal_url = f'wss://{signal_host}/ws'


In [ ]:
import asyncio
from scripts.check_signaling_vps import probe
await probe(signal_url, timeout=20)


**鍵を生成**: 出力の `Public key (share this)` をMacに渡します。秘密鍵は Colab ランタイムの中だけにあります。

In [ ]:
from pathlib import Path
from patchlib.roomid import auto_identity
key = Path('/content/kcia/private/colab.pem')
peer, signer, public, verifier = auto_identity(key)
print('Colab peer ID:',peer)
print('Colab PUBLIC KEY (share with Mac):',public)


**Macの公開鍵・共有ルームIDを入力**: Mac 側でも `scripts/check_wan_p2p.py identity --key private/mac.pem` を実行し、公開鍵のみ入手してください。ランダムroom-idは両端で同じものを使用します。Macでも Colabでも、どちらを先に開始しても構いません。

In [ ]:
from patchlib.roomid import validate_room
mac_public = input('Mac public key (base64; NOT private PEM): ').strip()
room_id = validate_room(input('Shared random room ID (24-64 chars): ').strip())
turn_only = input('Force TURN relay candidates on BOTH hosts? (y/N): ').strip().lower() == 'y'
print('Start the corresponding Mac connect command now; run the final cell within ~180 sec.')


In [ ]:
from argparse import Namespace
from scripts.check_wan_p2p import connect
args = Namespace(key=key, ws=signal_url, token="", room_id=room_id,
                 remote_public_key=mac_public, turn_only=turn_only,
                 turn_transport='tcp', timeout=180)
await connect(args)


## 任意: Colabで実PTを開始（上のハンドシェイクがPASSしてから）

Macで同一の `TRAIN_ROOM`, `HF_SHA`, モデル、rank、steps等を設定して `train-live` を起動してください。Colabでは CPU でなく CUDA を選べます。**Mac/Colab の状態ディレクトリ・秘密鍵は別々です。** Colabの学習依存ライブラリをインストールします。


In [ ]:
%pip -q install -r requirements-patches.txt


In [ ]:
import os
# Reuse the TURN/TCP route which passed on the VPS/Colab pilot.
os.environ['KCIA_TURN_TRANSPORT'] = 'auto'
os.environ['KCIA_FORCE_TURN_RELAY'] = '0'
os.environ.setdefault('KCIA_SNAPSHOT_RATE_MIB', '1.0')
os.environ.setdefault('KCIA_PATCH_RATE_MIB', '1.0')
import sys, subprocess
from pathlib import Path
from patchlib.roomid import validate_room
hf_sha = input('Pinned SmolLM2 revision (same 40-hex SHA as Mac): ').strip()
assert len(hf_sha)==40 and all(x in '0123456789abcdef' for x in hf_sha.lower())
train_room = validate_room(input('TRAIN_ROOM from Mac prepare (not the handshake room): ').strip())
print('Colab will automatically generate its OWN train-live Ed25519 identity beside its journal.')
print('Use a GPU runtime. Start the Mac trainer now, then run this cell.')
cmd=[sys.executable, 'scripts/start_wan_pt.py','run','--role','colab',
     '--room-id',train_room,'--revision',hf_sha,'--ws',signal_url,'--rounds','0',
     '--local-steps','32','--snapshot-priority','training']
print('Launching Colab PT. Stop with notebook Interrupt after confirming remote updates.')
subprocess.run(cmd,cwd='/content/kcia',check=True)
